In [ ]:
# Fix lỗi KeyError: 0 khi load tokenizer VietAI/vit5-base
# Lỗi này xảy ra do bug trong transformers khi xử lý vocab format
# Giải pháp: Patch tạm thời hoặc downgrade transformers

import sys
import transformers

print(f"Transformers version hiện tại: {transformers.__version__}")

# Thử patch tạm thời để fix lỗi KeyError: 0
try:
    from transformers.tokenization_utils_tokenizers import TokenizersBackend
    
    # Lưu hàm gốc
    original_convert = TokenizersBackend.convert_to_native_format
    
    def patched_convert_to_native_format(cls, trust_remote_code=False, **kwargs):
        """Patch để xử lý vocab có thể là dict thay vì list - fix KeyError: 0"""
        vocab = kwargs.get('vocab', None)
        if vocab is not None:
            # Nếu vocab là dict, không cần convert (giữ nguyên)
            if isinstance(vocab, dict):
                # Vocab dict không cần xử lý thêm
                pass
            # Nếu vocab là list, xử lý như bình thường
            elif isinstance(vocab, list) and len(vocab) > 0:
                if cls.model.__name__ == "Unigram":
                    # Chỉ xử lý nếu phần tử đầu là list/tuple
                    if isinstance(vocab[0], (list, tuple)):
                        vocab = [tuple(item) for item in vocab]
                        kwargs['vocab'] = vocab
        
        # Gọi hàm gốc với kwargs đã được xử lý
        try:
            return original_convert(cls, trust_remote_code=trust_remote_code, **kwargs)
        except KeyError as e:
            # Nếu vẫn bị KeyError, có thể do vocab là dict
            if 'vocab' in kwargs and isinstance(kwargs['vocab'], dict):
                # Vocab dict không cần convert, bỏ qua bước này
                print("⚠ Vocab là dict, bỏ qua convert_to_native_format")
                return kwargs
            raise
    
    # Áp dụng patch
    TokenizersBackend.convert_to_native_format = classmethod(patched_convert_to_native_format)
    print("✓ Đã áp dụng patch cho TokenizersBackend.convert_to_native_format")
    
except Exception as e:
    print(f"⚠ Không thể patch: {e}")
    print("Sẽ thử các cách khác khi load tokenizer...")

print("="*50)

In [ ]:
import pandas as pd

df = pd.read_excel(r"/kaggle/working/data-dg-train/DATA_DG_work.xlsx")   # hoặc path đúng của bạn

assert {"content", "summary", "grade"}.issubset(df.columns)
df = df.dropna().reset_index(drop=True)


from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os

MODEL_NAME = "VietAI/vit5-base"

# Kiểm tra phiên bản transformers
import transformers
print(f"Transformers version: {transformers.__version__}")

# Thử load tokenizer với nhiều cách khác nhau để tránh lỗi KeyError: 0
tokenizer = None
error_messages = []

# Cách 1: Load với use_fast=False và trust_remote_code
try:
    print("Đang thử load tokenizer với use_fast=False, trust_remote_code=True...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False, trust_remote_code=True)
    print("✓ Load thành công với use_fast=False")
except Exception as e:
    error_messages.append(f"use_fast=False: {str(e)}")
    print(f"✗ Lỗi: {str(e)[:200]}")

# Cách 2: Load với use_fast=True và trust_remote_code
if tokenizer is None:
    try:
        print("Đang thử load tokenizer với use_fast=True, trust_remote_code=True...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)
        print("✓ Load thành công với use_fast=True")
    except Exception as e:
        error_messages.append(f"use_fast=True: {str(e)}")
        print(f"✗ Lỗi: {str(e)[:200]}")

# Cách 3: Load không có use_fast (mặc định)
if tokenizer is None:
    try:
        print("Đang thử load tokenizer mặc định...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        print("✓ Load thành công với cài đặt mặc định")
    except Exception as e:
        error_messages.append(f"default: {str(e)}")
        print(f"✗ Lỗi: {str(e)[:200]}")

# Cách 4: Load với local_files_only=False và force_download
if tokenizer is None:
    try:
        print("Đang thử load với force_download...")
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME, 
            local_files_only=False,
            force_download=False,
            trust_remote_code=True
        )
        print("✓ Load thành công")
    except Exception as e:
        error_messages.append(f"force_download: {str(e)}")
        print(f"✗ Lỗi: {str(e)[:200]}")

# Cách 5: Thử load với T5Tokenizer trực tiếp (vit5 dựa trên T5)
if tokenizer is None:
    try:
        print("Đang thử load với T5Tokenizer trực tiếp...")
        from transformers import T5Tokenizer
        tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
        print("✓ Load thành công với T5Tokenizer")
    except Exception as e:
        error_messages.append(f"T5Tokenizer: {str(e)}")
        print(f"✗ Lỗi: {str(e)[:200]}")

if tokenizer is None:
    print("\n" + "="*50)
    print("KHÔNG THỂ LOAD TOKENIZER!")
    print("="*50)
    print("Tất cả các cách đều thất bại:")
    for msg in error_messages:
        print(f"  - {msg}")
    print("\nVui lòng kiểm tra:")
    print("  1. Kết nối internet để download model")
    print("  2. Phiên bản transformers (có thể cần downgrade hoặc upgrade)")
    print("  3. Cache của HuggingFace có thể bị corrupt")
    raise RuntimeError("Không thể load tokenizer với bất kỳ cách nào")

print(f"\n✓ Tokenizer loaded thành công!")
print(f"Model max length: {tokenizer.model_max_length}")  # ~512 tokens

from datasets import Dataset

MAX_INPUT_LEN = 512

def preprocess(batch, max_target_len):
    prompts = [
        f"Tóm tắt văn bản cho học sinh lớp {g}: {c}"
        for c, g in zip(batch["content"], batch["grade"])
    ]

    model_inputs = tokenizer(
        prompts,
        truncation=True,
        padding="max_length",
        max_length=MAX_INPUT_LEN,
    )

    labels = tokenizer(
        batch["summary"],
        truncation=True,
        padding="max_length",
        max_length=max_target_len,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# Không cần tokenize ở đây vì sẽ tokenize lại trong objective với max_target_len khác nhau
# tokenized_ds = dataset.map(
#     preprocess,
#     batched=True, 
#     remove_columns=dataset["train"].column_names
# )

# print(tokenized_ds["train"].column_names)

# ============================================
# CELL 1: OPTUNA HYPERPARAMETER TUNING
# ============================================

import optuna
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
import torch

def objective(trial):

    # ===== Optuna search space =====
    max_target_len = trial.suggest_categorical("max_target_len", [192, 256, 320])
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [2, 4])
    grad_acc = trial.suggest_categorical("grad_acc", [4, 8])
    num_epochs = trial.suggest_int("num_train_epochs", 3, 6)
    weight_decay = trial.suggest_float("weight_decay", 0.01, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.05, 0.1)

    torch.cuda.empty_cache()
    import gc
    gc.collect()

    # ===== Tokenize lại vì MAX_TARGET_LEN thay đổi =====
    def preprocess_with_max_len(batch):
        return preprocess(batch, max_target_len)
    
    tokenized_ds = dataset.map(
        preprocess_with_max_len,
        batched=True,
        remove_columns=dataset["train"].column_names
    )

    args = TrainingArguments(
        output_dir=f"./optuna_trial_{trial.number}",
        evaluation_strategy="no",
        logging_strategy="steps",
        logging_steps=100,
        save_strategy="no",

        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_acc,
        num_train_epochs=num_epochs,

        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        lr_scheduler_type="linear",

        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,

        label_smoothing_factor=0.1,
        max_grad_norm=1.0,

        fp16=True,
        gradient_checkpointing=True,

        dataloader_num_workers=0,
        dataloader_pin_memory=False,

        report_to="none"
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_ds["train"],
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    trainer.train()

    # Lấy loss từ log_history một cách an toàn
    train_loss = float("inf")
    if trainer.state.log_history:
        # Tìm entry cuối cùng có "loss"
        for log_entry in reversed(trainer.state.log_history):
            if "loss" in log_entry:
                train_loss = log_entry["loss"]
                break

    # ===== Cleanup =====
    del model, trainer, data_collator, args
    torch.cuda.empty_cache()
    gc.collect()

    return train_loss

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

best_params = study.best_params
print("=" * 50)
print("BEST HYPERPARAMETERS:")
print("=" * 50)
for key, value in best_params.items():
    print(f"{key}: {value}")
print("=" * 50)